# Domestic Metadata And CLIP Analysis

In [ ]:
from pathlib import Path
import sys

import pandas as pd


def _find_repo_root(start: Path = Path.cwd()) -> Path:
    candidates = [start, *start.parents, start / "cgsid"]
    for candidate in candidates:
        if (candidate / "src" / "cgsid").exists():
            return candidate
    raise FileNotFoundError("Could not find the cgsid repository root")


ROOT = _find_repo_root()
sys.path.insert(0, str(ROOT / "src"))

from cgsid.analysis.tools import (
    RelationSpec,
    all_directed_js,
    build_clip_metadata_from_mappings,
    clip_accuracy_by_label,
    expected_relation_table,
    format_clip_accuracy_table,
    plot_value_distributions,
    print_relation_tabs,
    value_distribution_table,
)
from cgsid.core.config import Settings

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 80)


In [ ]:
DATASET = "domestic"
DATASET_DIR = Settings().dataset_dir(DATASET)
METADATA_PATH = DATASET_DIR / "metadata.csv"
CLIP_PATH = DATASET_DIR / "clip_scores.csv"
COLUMNS = ["object_1", "object_1_color", "pose", "environment", "day_time", "object_2", "object_2_color"]
EXPECTED_RELATIONS: list[RelationSpec] = [
    ("animal -> room", "object_1", "environment"),
    ("room -> ball presence", "environment", "object_2"),
    ("ball presence -> animal", "object_2", "object_1"),
]


## Metadata

In [ ]:
metadata_path = METADATA_PATH
if not metadata_path.exists():
    raise FileNotFoundError(f'Metadata file not found: {metadata_path}')
metadata = pd.read_csv(metadata_path).copy()
print(f'metadata: {metadata_path} rows={len(metadata)}')
metadata[COLUMNS].head()


## Metadata Distributions

In [ ]:
metadata_value_distributions = value_distribution_table(metadata[COLUMNS], "metadata")
display(metadata_value_distributions)

plot_value_distributions(
    metadata[COLUMNS],
    f"Ground-truth label distributions in the {DATASET} dataset",
    color="#4f83b6",
)


## Metadata Expected JS

In [ ]:
metadata_expected_js = expected_relation_table(metadata, EXPECTED_RELATIONS)
display(metadata_expected_js)
print_relation_tabs(metadata, EXPECTED_RELATIONS)

## Metadata Full JS For Appendix

In [ ]:
metadata_full_js = all_directed_js(metadata, COLUMNS)
display(metadata_full_js)


## CLIP

In [ ]:
CLIP_MAPPINGS: dict[str, dict[str, str]] = {
    "object_1": {
        "dog": "pet_animal_dog",
        "cat": "pet_animal_cat",
    },
    "object_1_color": {
        "dark": "contains_color_dark",
        "brown": "contains_color_brown",
        "white": "contains_color_white",
        "ginger": "contains_color_ginger",
        "gray": "contains_color_gray",
    },
    "pose": {
        "lying": "pet_position_lying",
        "standing": "pet_position_standing",
    },
    "environment": {
        "living_room": "pet_room_living_room",
        "kitchen": "pet_room_kitchen",
    },
    "day_time": {
        "day": "pet_time_day",
        "evening": "pet_time_evening",
    },
    "object_2": {
        "ball": "pet_state_playing_with_ball",
        "none": "pet_state_not_playing_with_ball",
    },
    "object_2_color": {
        "red": "contains_ball_color_red",
        "yellow": "contains_ball_color_yellow",
        "green": "contains_ball_color_green",
        "blue": "contains_ball_color_blue",
        "orange": "contains_ball_color_orange",
        "purple": "contains_ball_color_purple",
    },
}


def build_clip_metadata(clip: pd.DataFrame) -> pd.DataFrame:
    pred = build_clip_metadata_from_mappings(clip, CLIP_MAPPINGS)
    pred.loc[pred["object_2"] == "none", "object_2_color"] = "none"
    return pred


In [ ]:
clip_path = CLIP_PATH
if not clip_path.exists():
    clip = None
    print(f'clip: no file found at {clip_path}')
    print('No CLIP results found yet. Run 2_query_dataset_with_clip_domestic.py, then rerun from this cell.')
else:
    clip = pd.read_csv(clip_path)
    print(f'clip: {clip_path} rows={len(clip)}')
    clip_pred = build_clip_metadata(clip)
    display(clip_pred[COLUMNS].head())


## CLIP Distributions

In [ ]:
if clip is not None:
    clip_value_distributions = value_distribution_table(clip_pred[COLUMNS], "clip")
    display(clip_value_distributions)
    clip_accuracy_suffixes = {
        column: f"acc {100 * metadata[column].reset_index(drop=True).eq(clip_pred[column].reset_index(drop=True)).mean():.1f}%"
        for column in COLUMNS
    }
    clip_label_accuracy_suffixes = {}
    for column in COLUMNS:
        truth = metadata[column].reset_index(drop=True)
        pred = clip_pred[column].reset_index(drop=True)
        correct = truth.eq(pred)
        for value in sorted(truth.dropna().unique(), key=str):
            mask = truth.eq(value)
            if mask.any():
                clip_label_accuracy_suffixes[(column, value)] = f"acc {100 * correct[mask].mean():.1f}%"

    plot_value_distributions(
        clip_pred[COLUMNS],
        f"CLIP-predicted label distributions in the {DATASET} dataset",
        color="#c87941",
        title_suffixes=clip_accuracy_suffixes,
        value_suffixes=clip_label_accuracy_suffixes,
    )


## CLIP Expected JS

In [ ]:
if clip is not None:
    clip_expected_js = expected_relation_table(clip_pred, EXPECTED_RELATIONS)
    display(clip_expected_js)
    print_relation_tabs(clip_pred, EXPECTED_RELATIONS)

## CLIP Full JS For Appendix

In [ ]:
if clip is not None:
    clip_full_js = all_directed_js(clip_pred, COLUMNS)
    display(clip_full_js)


## Metadata vs CLIP Summary

In [ ]:
if clip is not None:
    comparison = metadata_expected_js.merge(clip_expected_js, on=['relation', 'condition_column', 'target_column'], suffixes=('_metadata', '_clip'))
    comparison['weighted_js_delta_metadata_minus_clip'] = comparison['weighted_js_metadata'] - comparison['weighted_js_clip']
    display(comparison)

    wjs_comparison = metadata_full_js.merge(clip_full_js, on=['relation', 'condition_column', 'target_column'], suffixes=('_metadata', '_clip'))
    wjs_summary = wjs_comparison[['relation', 'weighted_js_metadata', 'weighted_js_clip']].rename(columns={
        'weighted_js_metadata': 'wjs (raw)',
        'weighted_js_clip': 'wjs (clip)',
    })
    wjs_summary = (
        wjs_summary
        .assign(_raw_sort=wjs_summary['wjs (raw)'].round(6), _clip_sort=wjs_summary['wjs (clip)'].round(6))
        .sort_values(['_raw_sort', '_clip_sort'], ascending=False)
        .drop(columns=['_raw_sort', '_clip_sort'])
        .reset_index(drop=True)
    )
    display(wjs_summary.style.format({'wjs (raw)': '{:.6f}', 'wjs (clip)': '{:.6f}'}))